### 定义State的三种方式
LangGraph 支持三种主要的 State 定义方式：

1. **TypedDict 方式** - 简单直接，适合快速原型
2. **Pydantic BaseModel 方式** - 类型验证，适合生产环境
3. **Annotated + Reducer 方式** - 自定义合并逻辑，适合复杂场景

#### 1. TypedDict方式
- Python 标准库提供，无需额外依赖
- 语法简洁，易于理解
- 仅提供静态类型提示，不做运行时验证
- 状态更新采用简单的字典覆盖策略

In [ ]:
from typing import TypedDict
class ChatState1(TypedDict):
    """使用 TypedDict 定义的聊天状态"""
    messages: list[str]      # 消息列表
    user_name: str           # 用户名
    turn_count: int          # 对话轮次
    
'''
这会存在消息累加问题，因为TypedDict默认是覆盖策略，而不是追加策略

# 错误示范
def node(state: ChatState) -> ChatState:  
	state["messages"].append("新消息")  # ❌ TypedDict 不可变    
	return state

# 正确做法
def node(state: ChatState) -> ChatState:    
	return {"messages": state["messages"] + ["新消息"]}  # ✅
'''

#### 2. Pydantic BaseModel模式
- 提供运行时数据验证
- 支持默认值、字段约束
- 自动生成友好的错误信息
- 与 FastAPI 等框架无缝集成

In [ ]:
from pydantic import BaseModel, Field

class ChatState2(BaseModel):
    """使用 Pydantic 定义的聊天状态"""
    messages: list[str] = Field(default_factory=list, description="消息历史")
    user_name: str = Field(default="Guest", description="用户名")
    turn_count: int = Field(default=0, ge=0, description="对话轮次")

'''
问题：
- 需要额外安装 Pydantic
- 轻微的性能开销（验证成本）
- 状态合并仍是覆盖策略
- 消息累加需要手动处理
'''


#### 3. Annotated + Reducer(高级方式)
- **自定义状态合并逻辑**
- 支持消息自动追加（不再需要手动 `messages + [new]`）
- 可以定义复杂的状态更新规则
- LangGraph 官方推荐用于聊天应用

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from operator import add

class ChatState(TypedDict):
    """使用 Annotated + Reducer 的聊天状态"""
    # Annotated[类型, Reducer函数]
    # add 是 operator.add，实现列表拼接
    messages: Annotated[list[str], add]
    user_name: str
    turn_count: int